# 01. 제주데이터허브 데이터 수집
**목적**: 이상 신호 탐지에 쓸 일 단위 신호(유동인구·와이파이·버스)와 맥락(기상·입도객·행사), 위치 기준(정류소·AP)을 허브 Open API로 받는다.

**허브 API 규칙 (2026-11-10 확인)**
- 호출: `open.jejudatahub.net/api/proxy/{데이터별 ID}/{프로젝트키}` — 키는 `.env`에만 저장
- 페이지: `number`, `limit`(최대 100) → 응답 `{totCnt, hasMore, data}`
- 일일 쿼터: **데이터별 1,000회** (프로젝트 화면에서 확인). 초과하면 서버가 **HTTP 403**
- 수집기(`app/ingest/hub_client.py`)는 쿼터를 데이터별로 세고, 멈춘 페이지부터 다음 날 이어받는다

**수집 중 발견한 문제와 해결**
| 문제 | 원인 | 해결 |
|---|---|---|
| 와이파이 AP그룹별 일일(593) 403 | 2020년 기술문서의 API ID가 바뀜 | 허브 상세 페이지의 현재 ID로 교체 → 1,420/1,420행 확인 |
| AP 정보(618) 실행마다 행 구성이 다름 | 페이지 정렬이 고정되지 않음(3,049행 중 고유 1,960 / 1,838) | 같은 요청을 반복해 고유 키로 합침 → 8회 만에 2,615개로 수렴 |
| 쿼터 초과 시 403을 오류로 재시도 | 403의 의미 미확인 | 403 = 쿼터 초과로 처리, 재시도하지 않고 이어받기 지점 저장 |
| 유동인구(583)·정류소별 버스(613) | 같은 날 2회 받아 비교 | 고유 키 수 = totCnt(688, 2,012) → 정렬 안정 확인 |

In [1]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
pd.set_option("display.max_colwidth", 60)
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False

## 수집 작업 정의 (`app/ingest/jobs.py`)

In [2]:
from app.ingest.jobs import JOBS
pd.DataFrame([{"작업": j.name, "허브 데이터": j.hub_data_id, "데이터명": j.title, "방식": j.mode, "고정 조건": j.extra or ""} for j in JOBS.values()])

,작업,허브 데이터,데이터명,방식,고정 조건
0,pop_domestic,583,읍면동 단위 내국인 유동인구,range,
1,pop_foreign_short,589,읍면동 단위 외국인 단기체류 유동인구,range,
2,wifi_ap_daily,593,와이파이 AP그룹별 일일 사용량,daily,
3,wifi_users_daily,586,와이파이 일일 사용자수,daily,
4,wifi_ap_info,618,제주 공공와이파이 AP 정보,daily,
5,bus_station_daily,613,일일 정류소별 버스 이용자 정보,range,"{'moveType': '승차', 'priceType': '일반', 'payType': '카드'}"
6,bus_total_daily,617,버스 일일 승객수 데이터,range,
7,bus_stations,612,버스 정류소 기본 정보,once,
8,bus_routes,611,버스 노선 기본 정보,once,
9,weather_daily,579,일별 기상 정보,daily,


## 수집 계획 (`app/ingest/plan.py`) — 주 분석 기간 2024-04-01 ~ 2025-03-31, 저비용 신호는 최근까지

In [3]:
from app.ingest.plan import PLAN
pd.DataFrame(PLAN, columns=["작업", "시작", "끝"])

,작업,시작,끝
0,bus_stations,2025-03-31,2025-03-31
1,bus_routes,2025-03-31,2025-03-31
2,weather_stations,2025-03-31,2025-03-31
3,visitors_purpose,2024-04-01,2025-07-31
4,visitors_country,2024-04-01,2025-07-31
5,bus_total_daily,2024-04-01,2026-11-05
6,wifi_users_daily,2024-04-01,2026-11-05
7,weather_daily,2024-04-01,2025-04-30
8,pop_domestic,2024-04-01,2025-03-31
9,wifi_ap_daily,2024-10-01,2025-03-31


## 수집 결과 (캐시 파일)

In [4]:
rows = []
for p in sorted(Path("data/cache").glob("*.parquet")):
    d = pd.read_parquet(p)
    date_col = next((c for c in ["baseDate", "dtYearMonth"] if c in d.columns), None)
    rng = (str(d[date_col].astype(str).min()), str(d[date_col].astype(str).max())) if date_col else ("-", "-")
    rows.append({"파일": p.stem, "행 수": len(d), "열 수": d.shape[1], "기간 시작": rng[0], "기간 끝": rng[1]})
pd.DataFrame(rows)

,파일,행 수,열 수,기간 시작,기간 끝
0,bus_routes,745,9,-,-
1,bus_stations,4352,8,-,-
2,bus_total_daily,6635,7,20240401,20261105
3,visitors_country,192,5,202404,202507
4,visitors_purpose,144,5,202404,202507
5,weather_daily,12176,13,20240401,20250330
6,weather_stations,42,13,-,-
7,wifi_ap_info,2615,10,20250331,20250331
8,wifi_users_daily,929,3,20240401,20261105


## 진행 중·남은 수집 (데이터별 오늘 사용한 호출 수)

In [5]:
from app.ingest.jobs import JOBS
import datetime as dt
today = dt.date.today().strftime("%Y%m%d")
by_api = {j.api_id: j.name for j in JOBS.values()}
used = []
for f in Path("data/state").glob(f"quota_{today}_*.txt"):
    api = f.stem.split("_", 2)[2]
    used.append({"작업": by_api.get(api, "(이전 ID)"), "오늘 호출 수": int(f.read_text() or 0), "남은 쿼터": max(0, 1000 - int(f.read_text() or 0))})
pd.DataFrame(used).sort_values("오늘 호출 수", ascending=False).reset_index(drop=True)

,작업,오늘 호출 수,남은 쿼터
0,wifi_ap_daily,973,27
1,wifi_users_daily,949,51
2,pop_domestic,895,105
3,bus_station_daily,447,553
4,weather_daily,395,605
5,wifi_ap_info,279,721
6,bus_total_daily,94,906
7,bus_stations,44,956
8,bus_routes,8,992
9,(이전 ID),3,997
